In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

TIMESTAMP_RUN = "20260827182159"
RAW_DATA_PATH = "../data/raw/sms_spam_indo.csv"
EVAL_DIR = f"../results/evaluation/{TIMESTAMP_RUN}-evaluation"

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)

plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_raw, x='Kategori', order=['ham', 'spam'], palette=['#4C72B0', '#C44E52'])
plt.title('Distribusi Kelas Dataset SMS Asli', fontweight='bold', pad=15)
plt.ylabel('Jumlah Pesan')
plt.xlabel('Kategori')

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='baseline', fontsize=11, color='black', xytext=(0, 5), textcoords='offset points')
plt.show()

In [ ]:
eval_files = glob.glob(f"{EVAL_DIR}/*.csv")
eval_files = [f for f in eval_files if "Summary" not in f]

data_length = []

spam_asli = df_raw[df_raw['Kategori'] == 'spam']['Pesan'].dropna()
for text in spam_asli:
    data_length.append({'Model': 'Original', 'Word_Count': len(str(text).split())})

for file in eval_files:
    filename = os.path.basename(file)
    model_name = filename.split('_zero-shot')[0].split('_few-shot')[0].split('_role-prompting')[0] 
    
    df_eval = pd.read_csv(file)
    for text in df_eval['Pesan'].dropna():
        data_length.append({'Model': model_name, 'Word_Count': len(str(text).split())})

df_length = pd.DataFrame(data_length)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_length, x='Model', y='Word_Count', showfliers=False, palette="Set2")
plt.title('Distribusi Jumlah Kata: Teks Asli vs Hasil Augmentasi LLM', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Jumlah Kata')
plt.xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
summary_file = os.path.join(EVAL_DIR, "Summary_TextQuality.csv")
df_summary = pd.read_csv(summary_file)

df_summary['Model_Teknik'] = df_summary['Model'] + " (" + df_summary['Teknik'] + ")"

df_summary = df_summary.sort_values(by='Cosine_Sim', ascending=False)

plt.figure(figsize=(12, 8))
ax1 = sns.barplot(data=df_summary, x='Cosine_Sim', y='Model_Teknik', color='#55A868', label='Cosine Similarity')

ax2 = sns.barplot(data=df_summary, x='BERT_score', y='Model_Teknik', color='#4C72B0', alpha=0.5, label='BERTScore')

plt.title('Kualitas Semantik Teks Sintetis', fontweight='bold')
plt.xlabel('Skor Semantik (0 - 1.0)')
plt.ylabel('')
plt.legend(loc='lower left')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
CLASS_DIR = f"../results/classification/{TIMESTAMP_RUN}-classification"
class_summary_file = os.path.join(CLASS_DIR, "Summary_Classification.csv")

df_class = pd.read_csv(class_summary_file)

df_class_sorted = df_class.sort_values(by=['F1-Score', 'ROC-AUC'], ascending=[False, False]).reset_index(drop=True)

display(df_class_sorted.style.highlight_max(subset=['Akurasi', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'], color='lightgreen', axis=0)
                       .format("{:.4f}", subset=['Akurasi', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']))

In [ ]:
top_models = df_class_sorted.head(5)['Dataset'].tolist()
if "Baseline_Data_Asli" not in top_models:
    top_models.append("Baseline_Data_Asli")

df_vis = df_class[df_class['Dataset'].isin(top_models)]

df_melted = df_vis.melt(id_vars='Dataset', 
                        value_vars=['Akurasi', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
                        var_name='Metrik', value_name='Skor')

plt.figure(figsize=(14, 7))
sns.barplot(data=df_melted, x='Metrik', y='Skor', hue='Dataset', palette='Set2')
plt.title('Komparasi Performa Klasifikasi: Baseline vs Augmentasi LLM', fontweight='bold', pad=15)
plt.ylabel('Skor (0.0 - 1.0)')
plt.xlabel('')
plt.ylim(0.7, 1.02)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier
import numpy as np

BEST_DATASET_NAME = df_class_sorted.iloc[0]['Dataset']
best_file_path = f"../data/augmented/{TIMESTAMP_RUN}/merged/{BEST_DATASET_NAME}.csv"

df_base = pd.read_csv(RAW_DATA_PATH)
df_best = pd.read_csv(best_file_path).dropna(subset=['Pesan', 'Kategori'])

def get_predictions(df):
    vec = TfidfVectorizer(max_features=5000)
    X = vec.fit_transform(df['Pesan']).toarray()
    y = df['Kategori'].map({'ham': 0, 'spam': 1}).values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    model = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
    model.fit(X_train, y_train)
    return y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1]

y_test_base, y_pred_base, y_prob_base = get_predictions(df_base)
y_test_best, y_pred_best, y_prob_best = get_predictions(df_best)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_test_base, y_pred_base), annot=True, fmt='d', cmap='Blues', ax=ax[0], cbar=False)
ax[0].set_title('Baseline (Data Asli)', fontweight='bold')
ax[0].set_ylabel('Aktual')
ax[0].set_xlabel('Prediksi')
ax[0].xaxis.set_ticklabels(['Ham', 'Spam']); ax[0].yaxis.set_ticklabels(['Ham', 'Spam'])

sns.heatmap(confusion_matrix(y_test_best, y_pred_best), annot=True, fmt='d', cmap='Greens', ax=ax[1], cbar=False)
ax[1].set_title(f'Augmentasi Terbaik ({BEST_DATASET_NAME})', fontweight='bold')
ax[1].set_ylabel('Aktual')
ax[1].set_xlabel('Prediksi')
ax[1].xaxis.set_ticklabels(['Ham', 'Spam']); ax[1].yaxis.set_ticklabels(['Ham', 'Spam'])

plt.suptitle('Reduksi False Negatives melalui Augmentasi Data', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

fpr_base, tpr_base, _ = roc_curve(y_test_base, y_prob_base)
fpr_best, tpr_best, _ = roc_curve(y_test_best, y_prob_best)

plt.figure(figsize=(7, 6))
plt.plot(fpr_base, tpr_base, label=f'Baseline (AUC = {auc(fpr_base, tpr_base):.4f})', color='#4C72B0', lw=2)
plt.plot(fpr_best, tpr_best, label=f'Best LLM (AUC = {auc(fpr_best, tpr_best):.4f})', color='#55A868', lw=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Curve', fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score
from scipy.stats import wilcoxon

def get_cv_scores(df):
    vec = TfidfVectorizer(max_features=5000)
    X = vec.fit_transform(df['Pesan']).toarray()
    y = df['Kategori'].map({'ham': 0, 'spam': 1}).values
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
    
    recall_scores = []
    for train_idx, test_idx in cv.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        y_pred = model.predict(X[test_idx])
        recall_scores.append(recall_score(y[test_idx], y_pred))
    return recall_scores

print("Melakukan 5-Fold Cross Validation untuk Uji Statistik...")
recall_base = get_cv_scores(df_base)
recall_best = get_cv_scores(df_best)

stat, p_value = wilcoxon(recall_base, recall_best)

df_stats = pd.DataFrame({
    'Fold': [1, 2, 3, 4, 5],
    'Recall_Baseline': recall_base,
    'Recall_Best_LLM': recall_best
})

df_stats.loc['Rata-rata'] = df_stats.mean()
display(df_stats.style.format("{:.4f}"))

print(f"\nWilcoxon Test P-Value : {p_value:.5f}")
if p_value < 0.05:
    print("Kesimpulan: Peningkatan performa SIGNIFIKAN secara statistik (p < 0.05).")
else:
    print("Kesimpulan: Peningkatan performa TIDAK signifikan secara statistik (p >= 0.05).")